# 1 Notebook 01 - Storage Formats and Compression Codecs

This notebook compares the row-based legacy format (CSV) with the columnar format (Parquet), with and without compression, using the clean base from notebook 00. 

For each format and codec it measures the space on HDFS and then write time and measure the read time in Spark.

### Step 1 - Imports

In [5]:
import time
import random
import pandas as pd                                
import pyspark
from pyspark.sql import SparkSession, functions as F
from pyspark import StorageLevel
import os 

### 0.2 - Spark Session
 
 *  4 cores of the VM and 3 GB for the driver where used to run the spark on the Ubuntu Orable VM.
 *  Every format is written with 8 files so the number of files is the same for all of them.
 *  Each read is timed 5 times after one warm-up round. 

In [6]:
# Initializing the Spark session

N_PART = 8   # same number of output files for every format, so file count is not a hidden variable
N_REPS = 5   # timed repetitions per read, after one discarded warm-up round.

spark = (SparkSession.builder
         .master("local[4]")                                   # local mode using the 4 cores of the VM
         .appName("ca1-01-formats")
         .config("spark.driver.memory", "3g")                  # leaves room for the HDFS daemons and the OS in 7.3 GB
         .config("spark.sql.session.timeZone", "UTC")          # avoids the Dublin DST shift on 31/03 with timestamp_ntz
         .config("spark.sql.shuffle.partitions", N_PART)       # makes the orderBy below produce N_PART partitions
         .getOrCreate())
spark

### Step 3 - Paths and Size on HDFS

`hdfs_size` returns the real number of bytes stored on HDFS, the same value as `hdfs dfs -du -s`. `MODEL_COLS` are the only columns the network needs, used later to test reading just part of the table.

In [9]:
# Defining the paths and the HDFS size helper
BASE = "hdfs://localhost:9000/ca1" 
CLEAN = f"{BASE}/clean/yellow_2024q1"    
OUT = f"{BASE}/formats"                   
MODEL_COLS = ["tpep_pickup_datetime", "trip_distance", "PULocationID", "DOLocationID", "duration_s_true"]   # (filtering colums for the ML Part) 

hadoop = spark._jvm.org.apache.hadoop     # Hadoop Java API, reached through Spark's JVM gateway
fs = hadoop.fs.FileSystem.get(spark._jvm.java.net.URI(BASE), spark._jsc.hadoopConfiguration())

# calcule the total bytes of all files under an HDFS path
def hdfs_size(path):
    return fs.getContentSummary(hadoop.fs.Path(path)).getLength()

In [10]:
# time order is the natural TLC order and makes row order deterministic
base = spark.read.parquet(CLEAN).orderBy("tpep_pickup_datetime")
# check if every format is written from exactly the same rows
base.persist(StorageLevel.MEMORY_AND_DISK)                           
base.count()                                                         

26/10/06 16:41:42 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: hdfs://localhost:9000/ca1/clean/yellow_2024q1.
java.net.ConnectException: Call From BDSP/127.0.1.1 to localhost:9000 failed on connection exception: java.net.ConnectException: Connection refused; For more details see:  http://wiki.apache.org/hadoop/ConnectionRefused
	at java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
	at java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:77)
	at java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:45)
	at java.base/java.lang.reflect.Constructor.newInstanceWithCaller(Constructor.java:500)
	at java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:481)
	at org.apache.hadoop.util.dynamic.DynConstructors$Ctor.newInstanceChecked(DynConstructors.java:58)
	at org.a

Py4JJavaError: An error occurred while calling o61.parquet.
: java.net.ConnectException: Call From BDSP/127.0.1.1 to localhost:9000 failed on connection exception: java.net.ConnectException: Connection refused; For more details see:  http://wiki.apache.org/hadoop/ConnectionRefused
	at java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
	at java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:77)
	at java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:45)
	at java.base/java.lang.reflect.Constructor.newInstanceWithCaller(Constructor.java:500)
	at java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:481)
	at org.apache.hadoop.util.dynamic.DynConstructors$Ctor.newInstanceChecked(DynConstructors.java:58)
	at org.apache.hadoop.util.dynamic.DynConstructors$Ctor.newInstance(DynConstructors.java:70)
	at org.apache.hadoop.net.

In [7]:
# Checking the partitions and the size of the clean base
print("partitions:", base.rdd.getNumPartitions())

print("clean base on HDFS (MB):", round(hdfs_size(CLEAN) / 1e6, 1)) 

partitions: 8
clean base on HDFS (MB): 202.9


In [11]:
# Writing an untimed warm-up file
base.write.mode("overwrite").parquet(f"{OUT}/_warmup")
# liberate HDFS space
fs.delete(hadoop.fs.Path(f"{OUT}/_warmup"), True)   

NameError: name 'base' is not defined

In [12]:
# Naming to each format in a random order and measuring size and write time
FORMATS = [("csv", "none"), ("csv", "gzip"),("parquet", "none"), ("parquet", "snappy"), ("parquet", "gzip"), ("parquet", "zstd")]


order = FORMATS.copy() # create a cope os the formarts 
random.Random(1234).shuffle(order)   # creating a randon order to avoid format to be avoided. 

rows = []
for fmt, codec in order:
    path = f"{OUT}/{fmt}_{codec}"
    writer = base.write.mode("overwrite").option("compression", codec)   # overwrite lets the cell be run 2 more than 1 time. 
    
    t0 = time.perf_counter()                                             # it shows how long a piece of code takes to run b
    if fmt == "csv":
        writer.option("header", True).csv(path)                          # stores every value as text and keep the head 
    else:
        writer.parquet(path)                                             # Parquet stores schema and types inside the file
        
    t_write = time.perf_counter() - t0                                   
    rows.append({"format": fmt, "codec": codec, "size_mb": hdfs_size(path) / 1e6, "write_s": t_write})
    print(f"{fmt:8}{codec:8}{rows[-1]['size_mb']:10.1f} MB {t_write:8.1f} s")

sizes = pd.DataFrame(rows) 

## Use as a reference https://www.geeksforgeeks.org/python/time-perf_counter-function-in-python/

NameError: name 'base' is not defined

We already have significant findings here to include in the report. The uncompressed CSV is 1,042 MB, whereas the Parquet file using zstd is approximately seven times smaller (cite literature explaining why this occurs). This validates the choice of zstd as the codec (explain the name and the underlying scientific rationale in the report).

The gzip-compressed CSV (195.7 MB) and the Snappy-compressed Parquet file (195.5 MB) are virtually the same size; this demonstrates that compression alone does not account for the advantages of the columnar format. Instead, the benefits stem from the column-oriented layout, typed binary encoding, and the ability to read only the requested columns. Look for literature comparing data types (*ARTICLES*).

In [10]:
# Defining how each format is read
def read_format(fmt, path):
    if fmt == "csv":
        return spark.read.schema(base.schema).option("header", True).csv(path)   # explicit schema: no extra pass for type inference
    return spark.read.parquet(path)                                             # schema comes from the Parquet file footer

In [11]:
# Measuring the read time in rounds, with a new random order in every round
rng = random.Random(1234)
times = []
for rep in range(N_REPS + 1):
    order = [(fmt, codec, scan) for fmt, codec in FORMATS for scan in ["full", "model_cols"]]
    rng.shuffle(order)                                   # a new random order in every round removes the order bias
    for fmt, codec, scan in order:
        df = read_format(fmt, f"{OUT}/{fmt}_{codec}")    # reading is lazy: nothing is read until the action below
        if scan == "model_cols":
            df = df.select(*MODEL_COLS)                  # projection: Parquet can skip the other columns, CSV reads every full line
        t0 = time.perf_counter()
        df.write.format("noop").mode("overwrite").save() # noop sink forces a real read and decode but writes nothing
        dt = time.perf_counter() - t0
        if rep > 0:                                      # round 0 is a full warm-up round and is discarded
            times.append({"format": fmt, "codec": codec, "scan": scan, "rep": rep, "read_s": dt})
    print("round", rep, "done")

times = pd.DataFrame(times)

round 0 done


round 1 done


round 2 done


round 3 done


round 4 done


[Stage 143:>                                                        (0 + 4) / 4]

round 5 done


In [12]:
# Building the summary table
med = times.pivot_table(index=["format", "codec"], columns="scan", values="read_s", aggfunc="median")   # median resists outliers from background activity in the VM
iqr = times.pivot_table(index=["format", "codec"], columns="scan", values="read_s",
                        aggfunc=lambda s: s.quantile(0.75) - s.quantile(0.25))   # interquartile range, reported with the median

summary = (sizes.set_index(["format", "codec"])
           .join(med.add_prefix("read_median_s_"))
           .join(iqr.add_prefix("read_iqr_s_")))
summary["ratio_vs_csv"] = summary.loc[("csv", "none"), "size_mb"] / summary["size_mb"]   # how many times smaller than the uncompressed CSV
display(summary.round(2))

size_mb  write_s  read_median_s_full  \
format  codec                                          
csv     gzip     195.71    48.26               15.41   
parquet none     264.99    12.23                2.29   
        zstd     144.30    14.84                2.29   
        gzip     148.02    33.82                2.42   
csv     none    1042.07    48.47               13.15   
parquet snappy   195.46    11.82                2.10   

                read_median_s_model_cols  read_iqr_s_full  \
format  codec                                               
csv     gzip                        7.23             6.41   
parquet none                        0.88             0.56   
        zstd                        0.87             0.44   
        gzip                        0.96             1.03   
csv     none                        7.04             2.25   
parquet snappy                      0.78             0.06   

                read_iqr_s_model_cols  ratio_vs_csv  
format  codec                                        
csv     gzip                     1.16          5.32  
parquet none                     0.14          3.93  
        zstd                     0.13          7.22  
        gzip                     0.08          7.04  
csv     none                     0.79          1.00  
parquet snappy                   0.02          5.33

Aqui oa resultados foram salvos em 1 pasta chamada results para serem comparados no futuro. 

In [18]:
# Saving the results to an absolute path, so it works wherever the notebook is saved
RESULTS = os.path.expanduser("~/ca1/results")
os.makedirs(RESULTS, exist_ok=True)                          
print("notebook folder:", os.getcwd())                       # shows the folder where the notebook is running 
times.to_csv(f"{RESULTS}/01_read_times.csv", index=False)    # raw read times, kept for the report
summary.to_csv(f"{RESULTS}/01_summary.csv")
base.unpersist();                                             # frees the cached base before the next notebook

notebook folder: /home/hduser/ca1


In [17]:
# Stopping the Spark session
spark.stop() 